# セミナーII 第2回 実践：画像を見て質問に答える（VLM：Qwen3-VL）

画像と文章を入力として受け取り、文章で答えるモデルを **VLM**（Vision-Language Model）と呼びます。ChatGPT や Gemini に写真を見せて質問するときも、VLM が動いています。

ここでは、小さな VLM（**Qwen3-VL-2B**、約20億パラメータ）を Colab の GPU で動かします。チャットの画面ではふだん見えない部分を、自分で確かめるのが目的です。

- 画像は、何個のトークンになって言語モデルに入るのか
- 答えの文章は、どうやって1トークンずつ作られるのか（確率と温度）
- 研究で使うときに便利な使い方（位置を答えさせる、まとめて処理して表にする）
- もっともらしいのに間違った答え（ハルシネーション）が、どんなときに出るのか

**最初に**

1. メニューの「ファイル」→「ドライブにコピーを保存」で、自分のコピーを作ります。
2. 「ランタイム」→「ランタイムのタイプを変更」で **GPU**（T4 など）を選びます。
3. 上のセルから順に実行します（セルの左の ▶、または **Shift + Enter**）。

ログインや Google ドライブへのアクセスの許可は必要ありません。全体で約40分（うち計算 約5分）です。

- セルの右に入力欄やプルダウンが出るセルでは、そこで値を選んでから実行します（コードを書き換えても同じです）。
- 図や表を作る関数のセルは、コードを畳んであります。実行するだけでかまいません。
- 質問は英語でも日本語でも書けます（このノートブックの例は英語）。
- しばらく操作しないと接続が切れます。`NameError` が出たときは、1. から順に実行し直してください。

## 0. このノートブックの流れ

目安：約1分

| 節 | やること |
| --- | --- |
| 1. 準備 | ライブラリとモデルを読み込む |
| 2. 手法を知る | 画像がトークンになるまで。答えを1トークンずつ作るしくみ。学習のしかた |
| 3. 例の画像で試す | 画像の説明と、簡単な質問 |
| 4. 自分の画像で試す | 自分の写真に質問する |
| 5. やってみよう | 温度を変える、画像トークンの数を変える、もっともらしい誤り |
| 6. 活用 | 物の位置を答えさせて枠を描く、何枚もの画像をまとめて表にする |
| 7. まとめ | チャットの画面との違いと、注意すること |

## 1. 準備

目安：約5分（うち計算 約3分）

`device: cuda` と GPU の名前が表示されれば準備完了です。`device: cpu` と表示されたら、「ランタイム」→「ランタイムのタイプを変更」で GPU を選んで、もう一度実行してください（このノートブックは CPU では動作が非常に遅くなります）。

In [ ]:
import importlib.util
import io
import json
import re
import subprocess
import sys
import time

# Colab に入っていない、または古いライブラリを入れる
for package in ["transformers", "accelerate", "pooch"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from skimage import data
from transformers import AutoModelForImageTextToText, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32  # T4 は bfloat16 に対応していないので float16
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "（GPU を選んでください）")

Qwen3-VL-2B-Instruct を読み込み、質問する関数 `ask` を定義します。初回は重み（約4GB）をダウンロードするので、2〜3分かかります。`HF_TOKEN` に関する警告が表示されても無視してかまいません。

- モデル：[Qwen/Qwen3-VL-2B-Instruct](https://huggingface.co/Qwen/Qwen3-VL-2B-Instruct)（Apache-2.0）。Instruct は、質問に答えるように追加学習した版
- 技術報告：Qwen Team, "Qwen3-VL Technical Report," 2025（[arXiv:2511.21631](https://arxiv.org/abs/2511.21631)）

In [ ]:
VLM_ID = "Qwen/Qwen3-VL-2B-Instruct"
processor = AutoProcessor.from_pretrained(VLM_ID)
model = AutoModelForImageTextToText.from_pretrained(VLM_ID, dtype=dtype).to(device).eval()
IMAGE_TOKEN = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")  # 画像トークンの目印


def set_image_tokens(max_tokens=1280, min_tokens=64):
    # 1枚の画像を何トークンまでにするか。1トークン = 32×32 画素なので、画素数の上限・下限で指定する
    processor.image_processor.size = {"longest_edge": max_tokens * 32 * 32, "shortest_edge": min_tokens * 32 * 32}


def prepare(image, question):
    # 画像と質問を、モデルへの入力（トークンの列）にする
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": question}]}]
    return processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                         return_dict=True, return_tensors="pt").to(device)


def ask(image, question, max_new_tokens=200, temperature=0):
    # 質問して、答えの文章を返す。temperature=0 は、毎回一番確率の高いトークンを選ぶ（貪欲法）
    inputs = prepare(image, question)
    sampling = {"do_sample": False} if temperature == 0 else {"do_sample": True, "temperature": temperature, "top_k": 0, "top_p": 1.0}
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, **sampling)
    return processor.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)


set_image_tokens()
print(f"パラメータの数: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B、語彙の数: {len(processor.tokenizer):,}")

図や表を作る関数と、例の画像を用意します。

In [ ]:
# @title 図や表を作る関数と、例の画像（実行するだけでよい） { display-mode: "form" }
images = {  # scikit-image に入っている写真
    "astronaut": Image.fromarray(data.astronaut()),              # NASA（パブリックドメイン）
    "coffee": Image.fromarray(data.coffee()),                    # 撮影 Rachel Michetti（CC0）
    "motorcycle": Image.fromarray(data.stereo_motorcycle()[0]),  # Middlebury 2014 ステレオデータセット
    "rocket": Image.fromarray(data.rocket()),                    # SpaceX（パブリックドメイン）
    "cat": Image.fromarray(data.chelsea()),                      # 撮影 Stefan van der Walt（CC0）
    "coins": Image.fromarray(data.coins()).convert("RGB"),       # Brooklyn Museum（著作権の制限は知られていない）
}

# 5-2 で読み取らせる表の画像（第1回の実験の結果を、matplotlib で描いたもの）
TABLE = pd.DataFrame({
    "experiment": ["baseline", "no augmentation", "batch 16", "no pretrained", "ResNet-50", "lr x100", "epochs 8"],
    "epochs": [3, 3, 3, 3, 3, 3, 8],
    "lr": [0.001, 0.001, 0.001, 0.001, 0.001, 0.1, 0.001],
    "test_acc": [0.828, 0.843, 0.825, 0.129, 0.875, 0.028, 0.855],
})
fig, ax = plt.subplots(figsize=(6, 2.4))
ax.axis("off")
ax.table(cellText=[[r.experiment, r.epochs, r.lr, f"{r.test_acc:.3f}"] for r in TABLE.itertuples()],
         colLabels=list(TABLE.columns), loc="center").scale(1, 1.3)
buffer = io.BytesIO()
fig.savefig(buffer, dpi=200, bbox_inches="tight")
plt.close(fig)
images["table"] = Image.open(buffer).convert("RGB")


def show(image, title="", size=4):
    plt.figure(figsize=(size * image.width / max(image.size), size * image.height / max(image.size)))
    plt.imshow(image)
    plt.title(title)
    plt.axis("off")
    plt.show()


def next_token_table(image, question, steps=8, k=5):
    # 答えの最初の steps 個のトークンについて、確率の高い候補 k 個を表にする（貪欲法で進める）
    inputs = prepare(image, question)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=steps, do_sample=False, output_scores=True,
                                return_dict_in_generate=True)
    rows = []
    for step, scores in enumerate(output.scores):
        probs = scores[0].float().softmax(dim=-1)  # 語彙の全トークンの確率（合計1）
        top = probs.topk(k)
        rows.append({"step": step + 1, **{f"#{j + 1}": f"{processor.tokenizer.decode([int(i)])!r} {p:.3f}"
                                          for j, (p, i) in enumerate(zip(top.values, top.indices))}})
    return pd.DataFrame(rows).set_index("step")


def parse_json(text):
    # 答えの中の JSON を取り出す（```json ... ``` で囲まれていることがある）
    match = re.search(r"```(?:json)?\s*(.*?)```", text, re.S)
    try:
        return json.loads(match.group(1) if match else text)
    except (json.JSONDecodeError, AttributeError):
        print("JSON として読めませんでした。答えの文章（先頭）:\n", text[:300], "…")
        return None


def draw_boxes(image, boxes, title=""):
    # bbox_2d（0〜1000 の相対座標）を、画像の上に描く
    plt.figure(figsize=(8, 8 * image.height / image.width))
    plt.imshow(image)
    for k, box in enumerate(boxes):
        x1, y1, x2, y2 = [v / 1000 * s for v, s in zip(box["bbox_2d"], [image.width, image.height] * 2)]
        color = plt.cm.tab10(k % 10)
        plt.gca().add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=color, lw=2.5))
        plt.text(x1, y1 - 4, box.get("label", ""), color="white", backgroundcolor=color, fontsize=10)
    plt.title(title)
    plt.axis("off")
    plt.show()


def fit_size(image, max_side=1280):
    # 大きすぎる写真は縮小する
    image = image.convert("RGB")
    scale = max_side / max(image.size)
    if scale < 1:
        image = image.resize((round(image.width * scale), round(image.height * scale)), Image.LANCZOS)
    return image


print("例の画像:", list(images))

## 2. 手法を知る

目安：約8分

### 2-1. 画像がトークンになるまで

VLM は、**画像エンコーダ**（ViT）と、文章を作る**言語モデル**（LLM）をつないだものです。

| 段階 | 処理（Qwen3-VL） |
| --- | --- |
| ① 画像エンコーダ | 画像を 16×16 画素のパッチに分け、ViT（SigLIP-2）でパッチごとの特徴ベクトルにする |
| ② つなぐ部分 | となり合う 2×2 個のパッチの特徴ベクトルを、2層の MLP で1つにまとめ、言語モデルのトークンと同じ次元のベクトルにする。これが**画像トークン**（1個 = 32×32 画素） |
| ③ 言語モデル | 質問の文章のトークンと画像トークンを、1つの列にして入力する。言語モデルから見ると、画像も「トークンの並び」になる |
| ④ 答えを作る | 答えの文章を、1トークンずつ作る（2-2） |

下のセルで、宇宙飛行士の画像（512×512）と短い質問が、何個のトークンになるかを確かめます。

In [ ]:
inputs = prepare(images["astronaut"], "Describe this image in one sentence.")
t, h, w = inputs["image_grid_thw"][0].tolist()  # パッチの数（時間, 縦, 横）
n_image = int((inputs["input_ids"][0] == IMAGE_TOKEN).sum())
print(f"画像: 512×512 画素 → 16×16 画素のパッチが {h}×{w} = {h * w} 個 → 2×2 個ずつまとめて {n_image} 個の画像トークン")
print(f"言語モデルへの入力: 全部で {inputs['input_ids'].shape[1]} トークン（画像 {n_image}、文章など {inputs['input_ids'].shape[1] - n_image}）")

画像の大きさによって、画像トークンの数が変わります。このノートブックでは、1枚の画像を64〜1280トークンにしています（技術報告のコードの例と同じ範囲）。画像トークンが多いほど細かいところまで読めますが、計算は重くなります（5-2 で確かめます）。

### 2-2. 答えを1トークンずつ作る

言語モデルは、それまでのトークンの列から、**次のトークンの確率**を、語彙（約15万個のトークン）のすべてについて計算します。第1回の分類が37クラスの softmax だったのに対して、ここでは約15万クラスの softmax です。

1. 確率にもとづいて、次のトークンを1つ選ぶ
2. 選んだトークンを列の最後に足す
3. 終わりの記号が選ばれるまで、1. に戻る

下の表は、宇宙飛行士の画像を説明させたとき、答えの最初の8トークンで、それぞれ確率の高い候補5個を並べたものです。一番確率の高いものを選び続けています（貪欲法）。

In [ ]:
question = "Describe this image in one sentence."
next_token_table(images["astronaut"], question)

確率が1つに集中しているトークンと、いくつかの候補に分かれているトークン（woman / female / astronaut など）があります。分かれているところで別の候補を選ぶと、別の文章になります（5-1 の温度）。

### 2-3. どうやって学習したか

技術報告によると、Qwen3-VL は次の順に学習しています。

1. 画像エンコーダ（SigLIP-2）と言語モデル（Qwen3）は、それぞれ別に学習済みのものを使う
2. まず、つなぐ部分（②の MLP）だけを、画像と文章の組で学習する（ほかは固定）
3. 次に、全体を、画像と文章のデータ（1兆トークン規模）で学習する。学習のしかたは「次のトークンを当てる」分類で、Loss は交差エントロピー
4. 最後に、質問と答えの組で追加学習して、質問に答えるようにする（Instruct）

## 3. 例の画像で試す

目安：約3分

4枚の画像を説明させ、1枚には簡単な質問をします。1枚あたり数秒かかります。

In [ ]:
for name in ["astronaut", "coffee", "motorcycle"]:
    show(images[name], name, size=3)
    print(ask(images[name], "Describe this image in one or two sentences."), "\n")

show(images["rocket"], "rocket", size=3)
print(ask(images["rocket"], "Was this photo taken during the day or at night? Explain briefly."))

**見どころ**：説明は自然ですが、よく読むと、画像からは分からないことや、間違ったことが混ざっていることがあります。たとえば宇宙飛行士の写真で、人の名前や、ワッペンに書かれた文字を答えていたら、それは正しいでしょうか（この写真は NASA の Eileen Collins 飛行士です）。小さなモデルは、もっともらしい名前や文字を作ってしまうことがあります（5-3 でも扱います）。

## 4. 自分の画像で試す

目安：約5分（画像を用意する時間は除く）

`USE_UPLOAD` にチェックを入れて実行すると、アップロードのボタンが表示されます（Colab のみ）。アップロードした画像は、ファイル名（拡張子を除く）の名前で、5. と 6. でも使えます。`QUESTION` に質問を書きます。

In [ ]:
USE_UPLOAD = False  # @param {type:"boolean"}
QUESTION = "Describe this image in detail."  # @param {type:"string"}

if USE_UPLOAD:
    from google.colab import files

    for filename, content in files.upload().items():
        name = filename.rsplit(".", 1)[0]
        images[name] = fit_size(Image.open(io.BytesIO(content)))
        show(images[name], name)
        print(ask(images[name], QUESTION))
print("使える画像の名前:", list(images))

## 5. やってみよう

目安：約12分（うち計算 約2分）

### 5-1. 温度を変える

次のトークンを選ぶとき、確率の高いものだけでなく、確率に従ってランダムに選ぶこともできます（サンプリング）。**温度** $T$ は、softmax の前にスコアを $T$ で割る値です。

- $T = 0$：常に一番確率の高いトークンを選ぶ（貪欲法）。毎回同じ答えになる
- $T$ が小さい（0.7 など）：確率の高い候補に集中する。答えは少しずつ変わる
- $T$ が大きい（1.5 など）：確率が均等に近づき、低い確率のトークンも選ばれる。文章が崩れることがある

CLIP では類似度に100をかけていましたが、これは温度 0.01 にあたります。同じ質問を、温度ごとに3回ずつ答えさせます。

In [ ]:
IMAGE = "cat"  # @param ["astronaut", "coffee", "motorcycle", "rocket", "cat", "coins"] {allow-input: true}
TEMPERATURES = [0, 0.7, 1.5]

for temperature in TEMPERATURES:
    print(f"--- temperature {temperature}")
    for _ in range(3):
        print(" ", ask(images[IMAGE], "Write a one-sentence caption for this photo.", max_new_tokens=40,
                         temperature=temperature))

チャットのサービスで、同じ質問をしても毎回少し違う答えが返ってくるのは、このサンプリングのためです（サービスごとの設定は公開されていないことが多いです）。モデルカードでは、画像の質問には温度 0.7 程度を勧めています。

### 5-2. 画像トークンの数を変える

第1回の実験の結果を表にした画像を、JSON に読み取らせます。1枚の画像に使うトークンの上限を変えると、画像はその数に収まるように縮小されてから入力されます。小さな文字が読めなくなると、どうなるでしょうか。正しい値と比べます。

In [ ]:
show(images["table"], "table (correct values)", size=6)
question = ("Read the table and output it as JSON: a list of objects with keys "
            "experiment, epochs, lr, test_acc. Output only JSON.")

for max_tokens in [16, 64, 256]:
    set_image_tokens(max_tokens=max_tokens, min_tokens=min(max_tokens, 64))
    n_image = int((prepare(images["table"], question)["input_ids"][0] == IMAGE_TOKEN).sum())
    start = time.time()
    rows = parse_json(ask(images["table"], question, max_new_tokens=400))
    print(f"--- 画像トークン {n_image} 個（{time.time() - start:.1f} 秒）")
    if rows:
        result = pd.DataFrame(rows).head(len(TABLE))
        # 正しい表と1行ずつ比べる（数値の列がすべて一致すれば OK）
        result["check"] = ["OK" if all(np.isclose(float(row[c]), TABLE.loc[i, c]) for c in ["epochs", "lr", "test_acc"])
                           else "NG" for i, row in result.iterrows()]
        display(result)
set_image_tokens()  # 元に戻す

画像トークンが少ないと、文字が読めないのに、**それらしい形の JSON と、もっともらしい数値**を出力することがあります。数値が少しずれているだけの行は、表だけを見ても気づけません。読み取った結果は、必ず元の画像と照らし合わせる必要があります。

### 5-3. もっともらしい誤り

硬貨の写真には、硬貨が24枚写っています。数を答えさせると、どうなるでしょうか。数を数えることは、VLM の苦手なことの1つです。自分の画像でも、細かい数、小さな文字、物の位置関係などを質問して、誤りを探してみましょう。

In [ ]:
show(images["coins"], "coins (24 coins)", size=4)
print("How many coins?  →", ask(images["coins"], "How many coins are in this image? Answer with a number."))

## 6. 活用：答えをプログラムで使う

目安：約8分（うち計算 約1分）

答えを決まった形式（JSON）で出力させると、プログラムでそのまま使えます。チャットの画面に1枚ずつ貼るのと違って、何百枚の画像にも同じ処理ができます。

### 6-1. 物の位置を答えさせて枠を描く

Qwen3-VL は、物の位置を、画像の左上を (0, 0)、右下を (1000, 1000) とした座標の枠（`bbox_2d`）で答えられます（技術報告による）。答えの JSON を読み取って、画像の上に枠を描きます。

In [ ]:
IMAGE = "coffee"  # @param ["astronaut", "coffee", "motorcycle", "rocket", "cat", "coins"] {allow-input: true}
OBJECTS = "the cup, the spoon, the saucer"  # @param {type:"string"}

image = images[IMAGE]
answer = ask(image, f'Locate {OBJECTS}. Output JSON: a list of {{"label": ..., "bbox_2d": [x1, y1, x2, y2]}}.')
print(answer)
boxes = parse_json(answer)
if boxes:
    draw_boxes(image, boxes, IMAGE)

### 6-2. 何枚もの画像をまとめて表にする

決まった項目を JSON で答えさせ、画像ごとに1行の表にします。画像を集めたあとで、内容ごとに整理したり、条件に合うものを選んだりするのに使えます。

In [ ]:
question = ('Answer in JSON with keys: "main_object" (a short noun phrase), "indoor_or_outdoor", '
            '"main_colors" (a list of up to 3 colors), "number_of_people" (an integer). Output only JSON.')

rows = []
for name, image in images.items():
    if name == "table":
        continue
    start = time.time()
    answer = parse_json(ask(image, question))
    if answer:
        rows.append({"image": name, **answer, "seconds": round(time.time() - start, 1)})
pd.DataFrame(rows)

## 7. まとめ

目安：約3分

| 分かったこと | 節 |
| --- | --- |
| VLM は、画像を ViT で画像トークン（1個 = 32×32 画素）にして、文章のトークンと一緒に言語モデルに入れる | 2-1 |
| 答えは、約15万個のトークンの softmax から、1トークンずつ選んで作る。温度で選び方が変わる | 2-2、5-1 |
| 画像トークンが足りないと、読めていないのに、もっともらしい答えを出すことがある | 5-2、5-3 |
| 答えを JSON で出力させると、位置の枠を描いたり、何枚もの画像を表にしたりできる | 6. |

**チャットの画面との違い**

- 中身を確かめられる：画像トークンの数、次のトークンの確率、温度を、自分で見て変えられる
- プログラムに組み込める：何百枚の画像に同じ質問をして、結果を表やファイルにできる
- 自分の計算機で動かせる：研究室の PC などで動かせば、データを外部のサービスに送らずに済む（ただし Colab は Google のサーバーで動いている）
- その代わり、このモデルは小さい（約20億パラメータ）。チャットのサービスの大きなモデルより、誤りが多い

**注意すること**：VLM の答えは、正しく見えても間違っていることがあります。研究で使うときは、一部を人が確認して、どのくらい正しいかを確かめてから使います。